# 00 — EDA

Sanity checks before modeling:
1. Shape, time range, sheet split
2. Top SKUs by revenue / volume
3. Country mix & customer activity
4. Total weekly demand and revenue (seasonality eyeball)
5. Christmas window check (UK gift-ware → November/December dominate)
6. Per-SKU demand class (Syntetos-Boylan: smooth / erratic / intermittent / lumpy)
7. Return-rate distribution
8. Price tier distribution

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.tools import (
    load_raw_data, split_sales_returns,
    aggregate_weekly_sku, return_rate_features,
    demand_classification, commercial_profile,
)

## 1. Shape, time range, sheet split

In [ ]:
df = load_raw_data(ROOT / "data" / "raw" / "online_retail_II.xlsx")
print("shape:", df.shape)
print("time range:", df['InvoiceDate'].min(), "→", df['InvoiceDate'].max())
print("\nrows per sheet:")
print(df['SourceSheet'].value_counts())
print("\nnulls:")
print(df.isna().sum())

In [ ]:
sales, returns = split_sales_returns(df)
print(f"sales rows: {len(sales):,}  |  unique SKUs: {sales['StockCode'].nunique():,}")
print(f"returns rows: {len(returns):,}  |  total returned units: {returns['Quantity'].sum():,}")
print(f"share of cleaned rows: {(len(sales)+len(returns))/len(df):.1%}")

## 2. Top SKUs by revenue / volume

In [ ]:
weekly = aggregate_weekly_sku(sales)
by_rev = weekly.groupby('StockCode')['Revenue'].sum().sort_values(ascending=False)
by_vol = weekly.groupby('StockCode')['Quantity'].sum().sort_values(ascending=False)

top_rev = by_rev.head(10).reset_index()
top_rev = top_rev.merge(
    sales.groupby('StockCode')['Description'].first().reset_index(),
    on='StockCode'
)
print("Top 10 SKUs by lifetime revenue:")
print(top_rev.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].barh(top_rev['Description'].str[:30][::-1], top_rev['Revenue'][::-1])
ax[0].set_title('Top 10 SKUs by Revenue'); ax[0].set_xlabel('£')
by_vol.head(10).plot.barh(ax=ax[1])
ax[1].set_title('Top 10 SKUs by Quantity'); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 3. Country mix & customer activity

In [ ]:
country_share = sales.groupby('Country')['Revenue'].sum().sort_values(ascending=False)
print(f"UK share of total revenue: {country_share['United Kingdom'] / country_share.sum():.1%}")
print("\nTop 10 countries by revenue:")
print(country_share.head(10).round(0))

n_customers = sales['Customer ID'].nunique()
n_anon = sales['Customer ID'].isna().sum()
print(f"\nunique customers (named): {n_customers:,}  |  anonymous rows: {n_anon:,} ({n_anon/len(sales):.1%})")

## 4. Total weekly demand & revenue — seasonality eyeball

In [ ]:
wk_total = weekly.groupby('Week').agg(Quantity=('Quantity', 'sum'), Revenue=('Revenue', 'sum'))
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
wk_total['Quantity'].plot(ax=ax[0]); ax[0].set_title('Total weekly quantity'); ax[0].set_ylabel('units')
wk_total['Revenue'].plot(ax=ax[1], color='tab:orange'); ax[1].set_title('Total weekly revenue (£)'); ax[1].set_ylabel('£')
plt.tight_layout(); plt.show()

## 5. Christmas window check
Gift-ware retailer → expect a sharp peak in Nov/Dec (and possibly mid-Oct if wholesale orders ship for the season).

In [ ]:
wk_total = wk_total.copy()
wk_total['month'] = wk_total.index.month
monthly = wk_total.groupby('month')['Quantity'].mean()
ax = monthly.plot.bar(figsize=(10, 4), title='Avg weekly quantity by month (across both years)')
ax.set_xlabel('month'); ax.set_ylabel('avg units / week')
plt.tight_layout(); plt.show()
print("Peak month:", int(monthly.idxmax()), "  multiplier vs Jan:", round(monthly.max() / monthly[1], 2))

## 6. Demand classification (Syntetos-Boylan)
Tells you which modeling family to favour:
- **smooth** → SARIMAX/Prophet shine
- **erratic** → Tweedie (LightGBM) / NegBin (DeepAR)
- **intermittent / lumpy** → Croston-SBA per cluster

In [ ]:
demand_cls = demand_classification(weekly)
print(demand_cls['demand_class'].value_counts(), "\n")
ax = demand_cls['demand_class'].value_counts().plot.bar(
    figsize=(8, 4), title='SKUs per demand class'
)
ax.set_ylabel('number of SKUs'); plt.tight_layout(); plt.show()
demand_cls.head()

## 7. Return-rate distribution
Which SKUs get returned often? Not the target, but a signal for the model and a quality flag.

In [ ]:
rr = return_rate_features(weekly, returns, windows=(13,))
lifetime = (
    rr.groupby('StockCode').agg(qty_sold=('Quantity', 'sum'), qty_returned=('qty_returned', 'sum'))
)
lifetime['return_rate'] = lifetime['qty_returned'] / lifetime['qty_sold'].replace(0, np.nan)
lifetime['return_rate'] = lifetime['return_rate'].fillna(0).clip(0, 1)
print(lifetime['return_rate'].describe(percentiles=[.5, .9, .99]).round(3))
ax = lifetime['return_rate'].plot.hist(bins=50, figsize=(10, 3), title='Lifetime return rate per SKU')
ax.set_xlabel('returned / sold'); plt.tight_layout(); plt.show()

## 8. Price tier distribution

In [ ]:
comm = commercial_profile(sales)
print(comm[['price_median', 'mean_basket_size', 'n_unique_customers', 'country_uk_share']].describe().round(2))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
comm['price_median'].clip(upper=comm['price_median'].quantile(0.99)).plot.hist(bins=60, ax=ax[0])
ax[0].set_title('Median price per SKU (clipped at p99)'); ax[0].set_xlabel('£')
comm['price_tier'].value_counts().sort_index().plot.bar(ax=ax[1])
ax[1].set_title('Price tier (quartile)'); ax[1].set_xlabel('tier')
plt.tight_layout(); plt.show()